In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    when,
    to_timestamp,
    min as spark_min,
    max as spark_max,
    sum as spark_sum
)

SOURCE_FILE = "ecommerce_clientes.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false"
}

EXPECTED_COLUMNS = [
    "id_cliente",
    "uuid_cliente",
    "nome",
    "sobrenome",
    "email",
    "senha_hash",
    "dt_cadastro",
    "dt_ultima_atualizacao"
]

KEY_COLUMN = "id_cliente"

adls_options = get_adls_options()

print(f"Arquivo analisado: {SOURCE_FILE}")
print(f"Caminho Raw: {SOURCE_PATH}")

In [0]:
df_raw_clientes = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

print("Leitura da Raw concluída.")
print(f"Total de linhas lidas: {df_raw_clientes.count()}")
print(f"Total de colunas: {len(df_raw_clientes.columns)}")

df_raw_clientes.printSchema()

In [0]:
actual_columns = df_raw_clientes.columns

missing_columns = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_columns = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

print(f"Colunas esperadas: {EXPECTED_COLUMNS}")
print(f"Colunas encontradas: {actual_columns}")
print(f"Colunas ausentes: {missing_columns}")
print(f"Colunas extras: {extra_columns}")

if missing_columns:
    raise Exception(f"Existem colunas obrigatórias ausentes: {missing_columns}")

print("Validação de colunas OK.")

In [0]:
df_datas_clientes = (
    df_raw_clientes
    .withColumn("dt_cadastro_ts", to_timestamp(col("dt_cadastro")))
    .withColumn("dt_ultima_atualizacao_ts", to_timestamp(col("dt_ultima_atualizacao")))
)

df_resumo_clientes = df_datas_clientes.select(
    count("*").alias("total_linhas"),
    countDistinct("id_cliente").alias("clientes_distintos"),
    spark_min("dt_cadastro_ts").alias("dt_cadastro_mais_antiga"),
    spark_max("dt_cadastro_ts").alias("dt_cadastro_mais_recente"),
    spark_min("dt_ultima_atualizacao_ts").alias("dt_ultima_atualizacao_mais_antiga"),
    spark_max("dt_ultima_atualizacao_ts").alias("dt_ultima_atualizacao_mais_recente")
)

display(df_resumo_clientes)

In [0]:
df_qualidade_clientes = df_datas_clientes.select(
    count("*").alias("total_linhas"),

    spark_sum(when(col("id_cliente").isNull(), 1).otherwise(0)).alias("id_cliente_nulo"),
    spark_sum(when(col("uuid_cliente").isNull(), 1).otherwise(0)).alias("uuid_cliente_nulo"),
    spark_sum(when(col("email").isNull(), 1).otherwise(0)).alias("email_nulo"),

    spark_sum(
        when(col("dt_cadastro").isNotNull() & col("dt_cadastro_ts").isNull(), 1)
        .otherwise(0)
    ).alias("dt_cadastro_invalida"),

    spark_sum(
        when(col("dt_ultima_atualizacao").isNotNull() & col("dt_ultima_atualizacao_ts").isNull(), 1)
        .otherwise(0)
    ).alias("dt_ultima_atualizacao_invalida"),

    (
        count("*") - countDistinct(KEY_COLUMN)
    ).alias("ids_clientes_duplicados")
)

display(df_qualidade_clientes)

In [0]:
from pyspark.sql.functions import lit

DATA_REFERENCIA_ANALISE = "2026-06-27"

df_clientes_analise = (
    df_datas_clientes
    .withColumn(
        "data_referencia_analise",
        lit(DATA_REFERENCIA_ANALISE).cast("timestamp")
    )
)

print(f"Data de referência da análise: {DATA_REFERENCIA_ANALISE}")

In [0]:
df_atualizacoes_futuras = df_clientes_analise.filter(
    col("dt_ultima_atualizacao_ts") > col("data_referencia_analise")
)

total_atualizacoes_futuras = df_atualizacoes_futuras.count()

print(f"Clientes com dt_ultima_atualizacao futura: {total_atualizacoes_futuras}")

display(
    df_atualizacoes_futuras
    .select(
        "id_cliente",
        "email",
        "dt_cadastro",
        "dt_ultima_atualizacao"
    )
    .orderBy(col("dt_ultima_atualizacao").desc())
    .limit(20)
)

In [0]:
df_atualizacao_antes_cadastro = df_clientes_analise.filter(
    col("dt_ultima_atualizacao_ts") < col("dt_cadastro_ts")
)

total_atualizacao_antes_cadastro = df_atualizacao_antes_cadastro.count()

print(f"Clientes com dt_ultima_atualizacao anterior ao dt_cadastro: {total_atualizacao_antes_cadastro}")

display(
    df_atualizacao_antes_cadastro
    .select(
        "id_cliente",
        "email",
        "dt_cadastro",
        "dt_ultima_atualizacao"
    )
    .orderBy("dt_cadastro")
    .limit(20)
)

In [0]:
from pyspark.sql.functions import year, month

df_cadastros_mes = (
    df_clientes_analise
    .withColumn("ano_cadastro", year(col("dt_cadastro_ts")))
    .withColumn("mes_cadastro", month(col("dt_cadastro_ts")))
    .groupBy("ano_cadastro", "mes_cadastro")
    .agg(
        count("*").alias("qtd_clientes")
    )
    .orderBy("ano_cadastro", "mes_cadastro")
)

display(df_cadastros_mes)

In [0]:
from pyspark.sql.functions import year, month

df_atualizacoes_mes = (
    df_clientes_analise
    .withColumn("ano_atualizacao", year(col("dt_ultima_atualizacao_ts")))
    .withColumn("mes_atualizacao", month(col("dt_ultima_atualizacao_ts")))
    .groupBy("ano_atualizacao", "mes_atualizacao")
    .agg(
        count("*").alias("qtd_clientes_atualizados")
    )
    .orderBy("ano_atualizacao", "mes_atualizacao")
)

display(df_atualizacoes_mes)

In [0]:
df_atualizacoes_futuras_mes = (
    df_atualizacoes_futuras
    .withColumn("ano_atualizacao", year(col("dt_ultima_atualizacao_ts")))
    .withColumn("mes_atualizacao", month(col("dt_ultima_atualizacao_ts")))
    .groupBy("ano_atualizacao", "mes_atualizacao")
    .agg(
        count("*").alias("qtd_clientes")
    )
    .orderBy("ano_atualizacao", "mes_atualizacao")
)

display(df_atualizacoes_futuras_mes)

In [0]:
display(
    df_atualizacoes_futuras
    .select(
        "id_cliente",
        "uuid_cliente",
        "nome",
        "sobrenome",
        "email",
        "dt_cadastro",
        "dt_ultima_atualizacao"
    )
    .orderBy(col("dt_ultima_atualizacao_ts").desc())
    .limit(30)
)

In [0]:
df_cadastro_clientes_atualizacao_futura = (
    df_atualizacoes_futuras
    .withColumn("ano_cadastro", year(col("dt_cadastro_ts")))
    .withColumn("mes_cadastro", month(col("dt_cadastro_ts")))
    .groupBy("ano_cadastro", "mes_cadastro")
    .agg(
        count("*").alias("qtd_clientes")
    )
    .orderBy("ano_cadastro", "mes_cadastro")
)

display(df_cadastro_clientes_atualizacao_futura)

In [0]:
from pyspark.sql.functions import datediff, avg, min as spark_min, max as spark_max

df_intervalo_atualizacao = (
    df_clientes_analise
    .withColumn(
        "dias_entre_cadastro_e_atualizacao",
        datediff(col("dt_ultima_atualizacao_ts"), col("dt_cadastro_ts"))
    )
)

display(
    df_intervalo_atualizacao.select(
        count("*").alias("total_clientes"),
        spark_min("dias_entre_cadastro_e_atualizacao").alias("menor_intervalo_dias"),
        spark_max("dias_entre_cadastro_e_atualizacao").alias("maior_intervalo_dias"),
        avg("dias_entre_cadastro_e_atualizacao").alias("media_intervalo_dias")
    )
)